# 05 — MLP models

MLP classifier training and evaluation. Uses `src.models.mlp` and `src.training.evaluation`.

In [1]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import logging
import numpy as np
import torch
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_reduced_dir, get_splits_dir, get_labels_dir, get_experiment_dir, get_experiment_output_dir, ensure_dir
from src.data import CLASS_NAMES, load_labels
from src.data.splits import load_splits
from src.models.mlp import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight
from src.training.evaluation import compute_metrics, compute_per_class_metrics, compute_relaxed_accuracy, compute_relaxed_per_class_metrics, save_metrics, save_confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# Toggles: which feature pipelines to run (set to True/False)
RUN_AUTOENCODER = True  # A1: autoencoder-reduced features
RUN_POOLED = True       # B1: pooled (kernel-informed) features

# MLP / training params (match A1 and B1 configs)
EPOCHS = 80
PATIENCE = 10
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 5e-4
HIDDEN_DIMS = [1024, 512, 256, 128]
DROPOUT = 0.2
LABEL_SMOOTHING = 0.1
NUM_CLASSES = len(CLASS_NAMES)  # 3 conditions + Sinus Rhythm (none)
print("MLP: epochs=%s patience=%s batch=%s lr=%s | %d classes: %s" % (EPOCHS, PATIENCE, BATCH_SIZE, LR, NUM_CLASSES, CLASS_NAMES))
print("Run autoencoder (A1): %s | Run pooled (B1): %s" % (RUN_AUTOENCODER, RUN_POOLED))

MLP: epochs=20 batch=256 lr=0.001 | 4 classes: ['AF', 'SVT', 'Sinus Brady', 'Sinus Rhythm']
Run autoencoder (A1): True | Run pooled (B1): True


In [ ]:
# Define device
def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = get_device()
print("Device: %s" % device)

In [2]:
# Load splits and labels (same order as saved train/test/val .npy)
print("Loading splits and labels...")
idx_train, idx_val, idx_test = load_splits()
try:
    y_full = load_labels()
except FileNotFoundError:
    print("labels.npy not found; loading from raw dataset...")
    from src.data import load_raw_dataset
    _, y_full = load_raw_dataset()
if y_full.ndim == 2:
    y_idx = np.argmax(y_full, axis=1)
    y_full_val = y_full[idx_val] if len(idx_val) > 0 else None
    y_full_test = y_full[idx_test]
else:
    y_idx = y_full
    y_full_val = None
    y_full_test = None
y_train = y_idx[idx_train]
y_test = y_idx[idx_test]
y_val = y_idx[idx_val] if len(idx_val) > 0 else None
print("y_train %s y_test %s" % (y_train.shape, y_test.shape))

Loading splits and labels...
labels.npy not found; loading from raw dataset...


2026-03-13 23:48:40,803 | INFO | Loaded cached dataset C:\Projects\DeepLearningProject\data\processed\chapman_wfdb_Xy.npz | X=(45150, 12, 5000) y=(45150, 4) | 22.58s


y_train (27090,) y_test (9030,)


In [ ]:
def train_mlp_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, input_dim, condition_name, y_full_val=None, y_full_test=None):
    """Train MLP on train set; evaluate on validation (if present) then on test. Save checkpoint and metrics.
    If y_full_val/y_full_test (multi-label) are provided, also report relaxed accuracy (correct if pred matches any true label)."""
    set_global_seed(SEED)
    print("Device: %s" % device)

    exp_dir = ensure_dir(get_experiment_dir(condition_name))
    ckpt_dir = ensure_dir(get_experiment_output_dir(condition_name, checkpoints=True))

    model = MLPClassifier(
        input_dim=input_dim,
        num_classes=NUM_CLASSES,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5)
    cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
    class_weight = torch.from_numpy(cw).float().to(device)
    n_train = len(X_train)
    labels_list = list(range(NUM_CLASSES))
    has_val = X_val is not None and y_val is not None and len(y_val) > 0
    best_val_f1 = -1.0
    epochs_no_improve = 0

    for ep in range(EPOCHS):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0
        n_b = 0
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            bx = torch.from_numpy(X_train[idx]).float().to(device)
            by = torch.from_numpy(y_train[idx]).long().to(device)
            logits = model(bx)
            loss = torch.nn.functional.cross_entropy(logits, by, weight=class_weight, label_smoothing=LABEL_SMOOTHING)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_b += 1
        if has_val:
            model.eval()
            with torch.no_grad():
                logits_val = model(torch.from_numpy(X_val).float().to(device))
                y_pred_val = logits_val.argmax(dim=1).cpu().numpy()
                metrics_val = compute_metrics(y_val, y_pred_val, task="multiclass", labels=labels_list)
            val_f1 = metrics_val["weighted_f1"]
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), ckpt_dir / "mlp_best.pt")
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print("  Early stopping at epoch %d (no val F1 improvement for %d epochs)" % (ep + 1, PATIENCE))
                break
            scheduler.step(val_f1)
        if (ep + 1) % 5 == 0 or ep == 0:
            print("  Epoch %d loss %.4f" % (ep + 1, total_loss / max(n_b, 1)))

    if has_val and (ckpt_dir / "mlp_best.pt").exists():
        model.load_state_dict(torch.load(ckpt_dir / "mlp_best.pt", map_location=device))
        print("  Loaded best checkpoint (val F1 %.4f)" % best_val_f1)
    torch.save(model.state_dict(), ckpt_dir / "mlp.pt")
    print("  Saved %s" % (ckpt_dir / "mlp.pt"))

    model.eval()
    all_metrics = {}
    labels_list = list(range(NUM_CLASSES))
    with torch.no_grad():
        if X_val is not None and y_val is not None and len(y_val) > 0:
            logits_val = model(torch.from_numpy(X_val).float().to(device))
            y_pred_val = logits_val.argmax(dim=1).cpu().numpy()
            metrics_val = compute_metrics(y_val, y_pred_val, task="multiclass", labels=labels_list)
            all_metrics["weighted_f1_val"] = metrics_val["weighted_f1"]
            per_class_val = compute_per_class_metrics(y_val, y_pred_val, labels=labels_list, target_names=CLASS_NAMES)
            all_metrics["per_class_val"] = per_class_val
            print("  --- Strict (one label per sample: argmax true) ---")
            print("  Validation weighted F1: %.4f" % metrics_val["weighted_f1"])
            print("  Validation per-class:")
            for p in per_class_val:
                print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
            if y_full_val is not None:
                print("  --- Relaxed (correct if pred in any true label) ---")
                rel_val = compute_relaxed_accuracy(y_full_val, y_pred_val, num_classes=NUM_CLASSES)
                all_metrics["relaxed_accuracy_val"] = rel_val
                per_class_relaxed_val, wf1_relaxed_val = compute_relaxed_per_class_metrics(y_full_val, y_pred_val, num_classes=NUM_CLASSES, target_names=CLASS_NAMES)
                all_metrics["per_class_relaxed_val"] = per_class_relaxed_val
                all_metrics["weighted_f1_relaxed_val"] = wf1_relaxed_val
                print("  Validation relaxed accuracy (any true label): %.4f" % rel_val)
                print("  Validation relaxed weighted F1: %.4f" % wf1_relaxed_val)
                print("  Validation relaxed per-class:")
                for p in per_class_relaxed_val:
                    print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
        logits_test = model(torch.from_numpy(X_test).float().to(device))
        y_pred_test = logits_test.argmax(dim=1).cpu().numpy()
    metrics_test = compute_metrics(y_test, y_pred_test, task="multiclass", labels=labels_list)
    per_class_test = compute_per_class_metrics(y_test, y_pred_test, labels=labels_list, target_names=CLASS_NAMES)
    all_metrics["weighted_f1"] = metrics_test["weighted_f1"]
    all_metrics["confusion_matrix"] = metrics_test.get("confusion_matrix", [])
    all_metrics["per_class"] = per_class_test
    if y_full_test is not None:
        print("  --- Relaxed (correct if pred in any true label) ---")
        rel_test = compute_relaxed_accuracy(y_full_test, y_pred_test, num_classes=NUM_CLASSES)
        all_metrics["relaxed_accuracy"] = rel_test
        per_class_relaxed, wf1_relaxed = compute_relaxed_per_class_metrics(y_full_test, y_pred_test, num_classes=NUM_CLASSES, target_names=CLASS_NAMES)
        all_metrics["per_class_relaxed"] = per_class_relaxed
        all_metrics["weighted_f1_relaxed"] = wf1_relaxed
        print("  Test relaxed accuracy (any true label): %.4f" % rel_test)
        print("  Test relaxed weighted F1: %.4f" % wf1_relaxed)
        print("  Test relaxed per-class:")
        for p in per_class_relaxed:
            print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
    save_metrics(all_metrics, exp_dir / "metrics.json")
    cm = np.array(metrics_test["confusion_matrix"])
    save_confusion_matrix(cm, exp_dir / "confusion_matrix.npy", path_png=exp_dir / "confusion_matrix.png", class_names=CLASS_NAMES)
    print("  --- Strict (one label per sample: argmax true) ---")
    print("  Test weighted F1: %.4f" % metrics_test["weighted_f1"])
    print("  Test per-class:")
    for p in per_class_test:
        print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
    return all_metrics

In [4]:
# A1: Autoencoder + MLP (load autoencoder-reduced features)
metrics_a1 = None
if RUN_AUTOENCODER:
    print("=== A1: Autoencoder + MLP ===")
    red_dir = get_reduced_dir() / "autoencoder"
    X_train_a1 = np.load(red_dir / "train.npy").astype(np.float32)
    X_test_a1 = np.load(red_dir / "test.npy").astype(np.float32)
    if (red_dir / "val.npy").exists():
        X_val_a1 = np.load(red_dir / "val.npy").astype(np.float32)
    else:
        X_val_a1 = None
    print("Loaded autoencoder features: train %s test %s (input_dim=%d)" % (X_train_a1.shape, X_test_a1.shape, X_train_a1.shape[1]))
    metrics_a1 = train_mlp_and_evaluate(X_train_a1, X_val_a1, X_test_a1, y_train, y_val, y_test, X_train_a1.shape[1], "A1_autoencoder_mlp", y_full_val=y_full_val, y_full_test=y_full_test)
else:
    print("Skipping A1 (autoencoder + MLP) — RUN_AUTOENCODER=False")

=== A1: Autoencoder + MLP ===
Loaded autoencoder features: train (27090, 256) test (9030, 256) (input_dim=256)


2026-03-13 23:48:41,835 | INFO | Global seed set to 0


  Epoch 1 loss 0.4631
  Epoch 5 loss 0.2378
  Epoch 10 loss 0.1693
  Epoch 15 loss 0.1223


2026-03-13 23:49:07,318 | INFO | Saved metrics to C:\Projects\DeepLearningProject\experiments\A1_autoencoder_mlp\metrics.json
2026-03-13 23:49:07,320 | INFO | Saved confusion matrix to C:\Projects\DeepLearningProject\experiments\A1_autoencoder_mlp\confusion_matrix.npy


  Epoch 20 loss 0.0927
  Saved C:\Projects\DeepLearningProject\experiments\A1_autoencoder_mlp\checkpoints\mlp.pt
  --- Strict (one label per sample: argmax true) ---
  Validation weighted F1: 0.9027
  Validation per-class:
    AF: P=0.890 R=0.919 F1=0.905 support=3554
    SVT: P=0.543 R=0.358 F1=0.431 support=302
    Sinus Brady: P=0.938 R=0.975 F1=0.956 support=3273
    Sinus Rhythm: P=0.917 R=0.850 F1=0.882 support=1901
  --- Relaxed (correct if pred in any true label) ---
  Validation relaxed accuracy (any true label): 0.7461
  Validation relaxed weighted F1: 0.8127
  Validation relaxed per-class:
    AF: P=0.487 R=0.908 F1=0.634 support=1968
    SVT: P=0.603 R=0.276 F1=0.379 support=434
    Sinus Brady: P=0.943 R=0.969 F1=0.956 support=3311
    Sinus Rhythm: P=0.922 R=0.779 F1=0.844 support=2083
  --- Relaxed (correct if pred in any true label) ---
  Test relaxed accuracy (any true label): 0.7464
  Test relaxed weighted F1: 0.8197
  Test relaxed per-class:
    AF: P=0.487 R=0.903 F

2026-03-13 23:49:07,444 | INFO | Saved confusion matrix figure to C:\Projects\DeepLearningProject\experiments\A1_autoencoder_mlp\confusion_matrix.png


  --- Strict (one label per sample: argmax true) ---
  Test weighted F1: 0.9070
  Test per-class:
    AF: P=0.895 R=0.914 F1=0.905 support=3575
    SVT: P=0.545 R=0.449 F1=0.493 support=296
    Sinus Brady: P=0.945 R=0.975 F1=0.960 support=3242
    Sinus Rhythm: P=0.918 R=0.857 F1=0.886 support=1917


In [5]:
# B1: Pooling + MLP (load pooled features)
metrics_b1 = None
if RUN_POOLED:
    print("=== B1: Pooling + MLP ===")
    red_dir = get_reduced_dir() / "pooled"
    X_train_b1 = np.load(red_dir / "train.npy").astype(np.float32)
    X_test_b1 = np.load(red_dir / "test.npy").astype(np.float32)
    if (red_dir / "val.npy").exists():
        X_val_b1 = np.load(red_dir / "val.npy").astype(np.float32)
    else:
        X_val_b1 = None
    print("Loaded pooled features: train %s test %s (input_dim=%d)" % (X_train_b1.shape, X_test_b1.shape, X_train_b1.shape[1]))
    metrics_b1 = train_mlp_and_evaluate(X_train_b1, X_val_b1, X_test_b1, y_train, y_val, y_test, X_train_b1.shape[1], "B1_pooling_mlp", y_full_val=y_full_val, y_full_test=y_full_test)
else:
    print("Skipping B1 (pooling + MLP) — RUN_POOLED=False")

2026-03-13 23:49:07,489 | INFO | Global seed set to 0


=== B1: Pooling + MLP ===
Loaded pooled features: train (27090, 176) test (9030, 176) (input_dim=176)
  Epoch 1 loss 0.5729
  Epoch 5 loss 0.2642
  Epoch 10 loss 0.1972
  Epoch 15 loss 0.1456


2026-03-13 23:49:32,930 | INFO | Saved metrics to C:\Projects\DeepLearningProject\experiments\B1_pooling_mlp\metrics.json
2026-03-13 23:49:32,932 | INFO | Saved confusion matrix to C:\Projects\DeepLearningProject\experiments\B1_pooling_mlp\confusion_matrix.npy


  Epoch 20 loss 0.1012
  Saved C:\Projects\DeepLearningProject\experiments\B1_pooling_mlp\checkpoints\mlp.pt
  --- Strict (one label per sample: argmax true) ---
  Validation weighted F1: 0.8967
  Validation per-class:
    AF: P=0.885 R=0.911 F1=0.898 support=3554
    SVT: P=0.513 R=0.258 F1=0.344 support=302
    Sinus Brady: P=0.950 R=0.961 F1=0.956 support=3273
    Sinus Rhythm: P=0.879 R=0.883 F1=0.881 support=1901
  --- Relaxed (correct if pred in any true label) ---
  Validation relaxed accuracy (any true label): 0.7446
  Validation relaxed weighted F1: 0.8089
  Validation relaxed per-class:
    AF: P=0.486 R=0.903 F1=0.632 support=1968
    SVT: P=0.579 R=0.203 F1=0.300 support=434
    Sinus Brady: P=0.956 R=0.956 F1=0.956 support=3311
    Sinus Rhythm: P=0.886 R=0.813 F1=0.848 support=2083
  --- Relaxed (correct if pred in any true label) ---
  Test relaxed accuracy (any true label): 0.7412
  Test relaxed weighted F1: 0.8108
  Test relaxed per-class:
    AF: P=0.484 R=0.897 F1=0.

2026-03-13 23:49:33,022 | INFO | Saved confusion matrix figure to C:\Projects\DeepLearningProject\experiments\B1_pooling_mlp\confusion_matrix.png


  --- Strict (one label per sample: argmax true) ---
  Test weighted F1: 0.8963
  Test per-class:
    AF: P=0.884 R=0.903 F1=0.893 support=3575
    SVT: P=0.540 R=0.297 F1=0.383 support=296
    Sinus Brady: P=0.959 R=0.957 F1=0.958 support=3242
    Sinus Rhythm: P=0.862 R=0.891 F1=0.876 support=1917


In [6]:
# Summary: overall and per-class (test set)
print("=== Summary ===")
if metrics_a1 is not None:
    print("A1 (autoencoder + MLP) test weighted F1: %.4f" % metrics_a1["weighted_f1"])
if metrics_b1 is not None:
    print("B1 (pooling + MLP) test weighted F1: %.4f" % metrics_b1["weighted_f1"])
if metrics_a1 is not None or metrics_b1 is not None:
    print("\nPer-class (test set):")
    n_cols = (3 if metrics_a1 else 0) + (3 if metrics_b1 else 0)
    hdr = "%-12s | " % "Class" + " ".join("%8s" % h for h in (["A1_P", "A1_R", "A1_F1"] if metrics_a1 else []) + (["B1_P", "B1_R", "B1_F1"] if metrics_b1 else []))
    print(hdr)
    print("-" * len(hdr))
    for i, name in enumerate(CLASS_NAMES):
        row = [name]
        if metrics_a1:
            a1 = metrics_a1.get("per_class", [])[i] if "per_class" in metrics_a1 else {}
            row.extend([a1.get("precision", 0), a1.get("recall", 0), a1.get("f1", 0)])
        if metrics_b1:
            b1 = metrics_b1.get("per_class", [])[i] if "per_class" in metrics_b1 else {}
            row.extend([b1.get("precision", 0), b1.get("recall", 0), b1.get("f1", 0)])
        print("%-12s | " % row[0] + " ".join("%8.3f" % v for v in row[1:]))
else:
    print("No runs selected (set RUN_AUTOENCODER and/or RUN_POOLED to True).")

=== Summary ===
A1 (autoencoder + MLP) test weighted F1: 0.9070
B1 (pooling + MLP) test weighted F1: 0.8963

Per-class (test set):
Class        |     A1_P     A1_R    A1_F1     B1_P     B1_R    B1_F1
--------------------------------------------------------------------
AF           |    0.895    0.914    0.905    0.884    0.903    0.893
SVT          |    0.545    0.449    0.493    0.540    0.297    0.383
Sinus Brady  |    0.945    0.975    0.960    0.959    0.957    0.958
Sinus Rhythm |    0.918    0.857    0.886    0.862    0.891    0.876
